# Notebook 3 — Threshold Analysis
Business-cost-aware threshold tuning.
Key question: Why is the default 0.5 threshold wrong for fraud detection?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys, os
sys.path.append(os.path.join('..', 'src'))
from fraud_detector import FraudDetectionPipeline, generate_synthetic_fraud_data
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

COST_FN = 500   # Missing a fraud: avg loss ($)
COST_FP = 15    # Blocking legit transaction: customer cost ($)

pipeline = FraudDetectionPipeline(output_dir='../reports')
df = generate_synthetic_fraud_data()
pipeline.load_and_explore(df)
pipeline.preprocess()
X_res, y_res = pipeline.apply_resampling()
pipeline.train_supervised_models(X_res, y_res)
print('Ready.')

## Business Cost Curve
Total cost = FN × $500 + FP × $15. We minimise this across all thresholds.

In [ ]:
# Use XGBoost (or first available model) probabilities
model_name = 'XGBoost' if 'XGBoost' in pipeline.results else list(pipeline.results.keys())[0]
y_prob = np.array(pipeline.results[model_name]['y_prob'])
y_true = pipeline.y_test

thresholds = np.linspace(0.01, 0.99, 300)
costs, precisions, recalls, f1s = [], [], [], []

for t in thresholds:
    y_pred = (y_prob >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    costs.append(fn * COST_FN + fp * COST_FP)
    precisions.append(precision_score(y_true, y_pred, zero_division=0))
    recalls.append(recall_score(y_true, y_pred))
    f1s.append(f1_score(y_true, y_pred))

optimal_idx = np.argmin(costs)
optimal_thresh = thresholds[optimal_idx]
min_cost = costs[optimal_idx]
baseline_cost = y_true.sum() * COST_FN

fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

# Cost curve
axes[0].plot(thresholds, costs, color='#E24B4A', linewidth=2, label='Total business cost')
axes[0].axhline(baseline_cost, color='gray', linestyle='--', linewidth=1.5,
                label=f'No-model baseline: ${baseline_cost:,}')
axes[0].axvline(optimal_thresh, color='#1D9E75', linestyle='--', linewidth=2,
                label=f'Optimal threshold: {optimal_thresh:.3f} (${min_cost:,})')
axes[0].axvline(0.5, color='#BA7517', linestyle=':', linewidth=2, label='Default threshold 0.5')
axes[0].set_ylabel('Total cost ($)', fontsize=11)
axes[0].set_title(f'{model_name} — Business Cost vs Threshold', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

# Precision / Recall / F1
axes[1].plot(thresholds, recalls, color='#378ADD', linewidth=2, label='Recall')
axes[1].plot(thresholds, precisions, color='#1D9E75', linewidth=2, label='Precision')
axes[1].plot(thresholds, f1s, color='#BA7517', linewidth=2, linestyle='--', label='F1 Score')
axes[1].axvline(optimal_thresh, color='#E24B4A', linestyle='--', linewidth=2,
                label=f'Optimal: {optimal_thresh:.3f}')
axes[1].axvline(0.5, color='gray', linestyle=':', linewidth=1.5, label='Default 0.5')
axes[1].set_xlabel('Decision Threshold', fontsize=11)
axes[1].set_ylabel('Score', fontsize=11)
axes[1].set_title('Precision / Recall / F1 vs Threshold', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/figures/threshold_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

savings = baseline_cost - min_cost
print(f'Optimal threshold:    {optimal_thresh:.3f}')
print(f'Min cost at optimal:  ${min_cost:,}')
print(f'Baseline (no model):  ${baseline_cost:,}')
print(f'Estimated savings:    ${savings:,}')

## Comparing Threshold 0.5 vs Optimal

In [ ]:
results_table = []
for label, thresh in [('Default (0.5)', 0.5), (f'Optimal ({optimal_thresh:.3f})', optimal_thresh)]:
    y_pred = (y_prob >= thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    cost = fn * COST_FN + fp * COST_FP
    results_table.append({
        'Threshold': label,
        'TP (caught)': tp, 'FN (missed)': fn, 'FP (false alarms)': fp,
        'Precision': f'{precision_score(y_true, y_pred, zero_division=0):.3f}',
        'Recall': f'{recall_score(y_true, y_pred):.3f}',
        'F1': f'{f1_score(y_true, y_pred):.3f}',
        'Total Cost ($)': f'${cost:,}',
    })

pd.DataFrame(results_table).set_index('Threshold').T

## Key Takeaways
- The optimal threshold is **lower than 0.5** — we flag more transactions as fraud
- This trades some precision (more false alarms) for higher recall (fewer missed frauds)
- Justified by the 33:1 cost asymmetry: missing fraud costs 33× more than a false alarm
- Always tune threshold using **business costs**, not arbitrary 0.5